# LangChain L11 — Level 10 — Human-in-the-loop
Even the finance role should not refund money unattended. The right design is not
"Agent, be careful" but an architectural pause:

```text
Agent  -->  "refund_customer(C002, 500)"  -->  PAUSE (interrupt)
                                                  |
                                       Human: approve / edit / reject
                                                  |
                                          Agent resumes
```

`HumanInTheLoopMiddleware` interrupts before the listed tools run. Because the agent state is
checkpointed (L6), the process can stop, wait minutes or days, and resume exactly there with the
human's decision. This is the mechanism behind every "approve this action" button in agent products.

### Step 1 — Configure which tools need approval

`interrupt_on` maps tool names to policies: `True` allows approve/edit/reject, `False` means
automatic. Read tools stay automatic; the write tool pauses.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

approval_policy = HumanInTheLoopMiddleware(interrupt_on={
    "get_customer": False,           # safe read: automatic
    "get_order": False,
    "refund_customer": True,         # money moves: a human decides
})

opspilot_hitl = create_agent(model=model, tools=[get_customer, get_order, refund_customer], system_prompt=OPSPILOT_PROMPT,
                             middleware=[approval_policy], checkpointer=InMemorySaver())

ticket = {"configurable": {"thread_id": "ticket-4711"}}
result = opspilot_hitl.invoke({"messages": [{"role": "user", "content": "Customer C002 was charged twice for order O1002. Issue a refund of 500 to C002."}]}, ticket)

interrupt = result["__interrupt__"][0]
print("PAUSED. The agent wants to run:")
for action in interrupt.value["action_requests"]:
    print("   ", action["name"], json.dumps(action["args"]))
print("allowed decisions:", interrupt.value["review_configs"][0]["allowed_decisions"])
print("refund ledger so far:", len(REFUND_LEDGER), "entries")

### Step 2 — Resume with a decision

The human's answer travels back as `Command(resume=...)` on the same thread. One decision per
pending action, in order. Try `approve`; then the same request on a new thread with `reject`,
which sends the model a message explaining why so it can respond to the user.

In [ ]:
resumed = opspilot_hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), ticket)
print("after APPROVE :", text_of(resumed["messages"][-1])[:120])
print("ledger        :", REFUND_LEDGER[-1])

ticket2 = {"configurable": {"thread_id": "ticket-4712"}}
opspilot_hitl.invoke({"messages": [{"role": "user", "content": "Customer C001 was charged twice for order O1001. Issue a refund of 120 to C001."}]}, ticket2)
resumed = opspilot_hitl.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Order O1001 shows a single charge. Do not refund; explain to the user."}]}), ticket2)
print("after REJECT  :", text_of(resumed["messages"][-1])[:140])
print("ledger size   :", len(REFUND_LEDGER), "(unchanged by the rejection)")

### Step 3 — Edit before approving

A reviewer may correct the arguments instead of rejecting outright: approve the refund, but for
the verified amount. `edit` replaces the action with the reviewer's version.

In [ ]:
ticket3 = {"configurable": {"thread_id": "ticket-4713"}}
paused = opspilot_hitl.invoke({"messages": [{"role": "user", "content": "Customer C002 was charged twice for order O1002. Issue a refund of 900 to C002."}]}, ticket3)
requested = paused["__interrupt__"][0].value["action_requests"][0]
print("requested :", requested["name"], requested["args"])

edited = {"name": "refund_customer", "args": {**requested["args"], "amount": 500.0, "reason": "duplicate charge verified on O1002"}}
resumed = opspilot_hitl.invoke(Command(resume={"decisions": [{"type": "edit", "edited_action": edited}]}), ticket3)
print("executed  :", REFUND_LEDGER[-1])
print("answer    :", text_of(resumed["messages"][-1])[:120])

### Recap

- **Problem seen:** a write tool executed the moment the model asked for it.
- **Layer added:** `HumanInTheLoopMiddleware` with an `interrupt_on` policy, checkpointed pauses, and `Command(resume=...)` decisions.
- **Evidence:** the refund only reached the ledger after an approve or an edit; the rejection left the ledger unchanged.